# Baseline RAG — SEC Filings

**Pipeline**: Query → Dense Retrieval (top-5) → LLM Generation

The simplest possible RAG baseline:
- Embed the raw query with `all-mpnet-base-v2`
- Retrieve top-5 chunks from the pre-built ChromaDB vector store
- Pass context + question to `llama-3.1-8b-instant` (dev) or `llama-4-scout-17b` (final)
- No query transformation, no metadata filtering, no reranking

Results saved to `./results/baseline_results.csv` for comparison with Advanced RAG.

In [2]:
print('hi')

hi


In [3]:
# Uncomment to install dependencies
# !pip install sentence-transformers chromadb groq python-dotenv pandas tqdm

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached groq-1.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.10.0-cp311-cp311-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached regex-2026.2.28-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dm-tree 0.1.9 requires absl-py>=0.6.1, which is not installed.
dm-tree 0.1.9 requires wrapt>=1.11.2, which is not installed.
tensorflow-datasets 4.9.9 requires absl-py, which is not installed.
tensorflow-datasets 4.9.9 requires pyarrow, which is not installed.
tensorflow-datasets 4.9.9 requires termcolor, which is not installed.
tensorflow-datasets 4.9.9 requires toml, which is not installed.
tensorflow-datasets 4.9.9 requires wrapt, which is not installed.
tensorflow-metadata 1.17.2 requires absl-py<3.0.0,>=0.9, which is not installed.


## 1. Configuration

In [4]:
CONFIG = {
    # --- Paths (relative to this notebook) ---
    "chroma_db_path": "../sec_rag_team_share/chroma_db",
    "chunks_path":    "../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl",
    "eval_path":      "../sec_rag_team_share/evaluation/GenAI Eval QA.csv",
    "results_dir":    "./results",

    # --- Models (same as langgraph_agentic_rag_sec_v2) ---
    "dense_model_name": "sentence-transformers/all-mpnet-base-v2",
    # Execution profile: 'dev' (llama-3.1-8b-instant) or 'final' (llama-4-scout-17b)
    "execution_profile": "dev",
    "groq_dev_model":   "llama-3.1-8b-instant",
    "groq_final_model": "meta-llama/llama-4-scout-17b-16e-instruct",
    "judge_model":      "llama-3.1-8b-instant",

    # --- Retrieval ---
    "dense_top_k":       5,
    "max_context_chars": 7000,

    # --- Generation ---
    "temperature_generator": 0.2,
    "temperature_judge":     0.0,
    "max_tokens_answer":     512,
    "max_tokens_judge":      256,

    # --- Evaluation ---
    "eval_split": "dev",   # switch to 'test' for final evaluation

    # --- Rate limiting (Groq hard cap ~30 RPM) ---
    "max_rpm":                    28,
    "inter_question_sleep_sec":   2.2,
    "llm_max_retries":            3,
    "llm_retry_base_delay_sec":   5,
}

## 2. Imports & Setup

In [5]:
import os
import json
import time
import threading
from collections import deque
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Set GROQ_API_KEY in your .env file"

# Resolve LLM model from profile
LLM_MODEL = (
    CONFIG["groq_final_model"]
    if CONFIG["execution_profile"] == "final"
    else CONFIG["groq_dev_model"]
)
print(f"Execution profile : {CONFIG['execution_profile']}")
print(f"LLM model         : {LLM_MODEL}")

c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Execution profile : dev
LLM model         : llama-3.1-8b-instant


## 3. Load Models

In [6]:
print("Loading embedding model...")
embed_model = SentenceTransformer(CONFIG["dense_model_name"])
print(f"  ok {CONFIG['dense_model_name']}")

groq_client = Groq(api_key=GROQ_API_KEY)
print("  ok Groq client ready")


class RateLimiter:
    """Keeps LLM requests under Groq's RPM cap."""
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= 60.0:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = 60.0 - (now - self._ts[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] sleeping {sleep_for:.1f}s ...")
                    time.sleep(sleep_for)
            self._ts.append(time.time())


rate_limiter = RateLimiter(CONFIG["max_rpm"])


def call_llm(
    messages: list,
    model: str = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
    json_mode: bool = False,
) -> str:
    """Call Groq LLM with retry and exponential back-off."""
    model = model or LLM_MODEL
    kwargs = dict(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            resp = groq_client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1}/{CONFIG['llm_max_retries']} failed ({e}). Retrying in {delay}s...")
            time.sleep(delay)
    raise RuntimeError(f"LLM call failed after {CONFIG['llm_max_retries']} attempts")

Loading embedding model...


c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jolen\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11704.27it/s]
MPNetModel LOAD RE

  ok sentence-transformers/all-mpnet-base-v2
  ok Groq client ready


## 4. Load ChromaDB Vector Store

In [7]:
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_db_path"])
collections = chroma_client.list_collections()
print(f"  Available collections: {[c.name for c in collections]}")
collection = chroma_client.get_collection(collections[0].name)
print(f"  ok Using '{collections[0].name}'  ({collection.count():,} chunks)")

Connecting to ChromaDB...
  Available collections: ['sec_filings']
  ok Using 'sec_filings'  (12,606 chunks)


## 5. Retrieval — Dense Search Only

In [ ]:
# def retrieve(query: str, top_k: int = None) -> list:
#     """
#     Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
#     Returns list of dicts: text, metadata, score, chunk_id.
#     """
#     k = top_k or CONFIG["dense_top_k"]
#     query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
#     results = collection.query(
#         query_embeddings=[query_vec],
#         n_results=min(k, collection.count()),
#         include=["documents", "metadatas", "distances", "ids"],
#     )
#     chunks = []
#     for doc, meta, dist, cid in zip(
#         results["documents"][0],
#         results["metadatas"][0],
#         results["distances"][0],
#         results["ids"][0],
#     ):
#         chunks.append({
#             "text":     doc,
#             "metadata": meta,
#             "score":    float(1.0 - dist),  # cosine distance -> similarity
#             "chunk_id": cid,
#         })
#     return chunks

In [14]:
def retrieve(query: str, top_k: int = None) -> list:
    """
    Dense retrieval: embed query -> cosine nearest-neighbor in ChromaDB.
    Returns list of dicts: text, metadata, score, chunk_id.
    """
    k = top_k or CONFIG["dense_top_k"]
    query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],  
    )

    chunks = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0], 
    ):
        chunks.append({
            "text": doc,
            "metadata": meta,
            "score": float(1.0 - dist),  # cosine distance -> similarity
            "chunk_id": cid,
        })

    return chunks

## 6. Generation

In [11]:
GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers -- include units (millions, billions, %), fiscal periods, and line item names. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)


def format_context(chunks: list, max_chars: int = None) -> str:
    """Format retrieved chunks into a numbered context string."""
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        header = f"[{i}] {m.get('ticker','?')} {m.get('form_type','?')} {m.get('filing_year','?')}"
        parts.append(f"{header}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return context[:max_chars]


def generate(query: str, chunks: list) -> str:
    """Generate an answer from retrieved chunks."""
    context  = format_context(chunks)
    user_msg = f"Question:\n{query}\n\nRetrieved context:\n{context}"
    return call_llm(
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=CONFIG["temperature_generator"],
        max_tokens=CONFIG["max_tokens_answer"],
    )

## 7. Full Baseline RAG Pipeline

In [22]:
def baseline_rag(query: str) -> dict:
    """
    Baseline RAG: Dense Retrieval -> LLM Generation.
    Returns: query, answer, chunks, context.
    """
    chunks  = retrieve(query)
    context = format_context(chunks)
    answer  = generate(query, chunks)

    # print(f'chunks: {chunks}')
    # print('\n\n\---------------')

    # print(f'context: {context}')
    # print('\n\n\---------------')

    # print(f'answer: {answer}')
    return {"query": query, "answer": answer, "chunks": chunks, "context": context}

### Quick Demo

In [18]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
# demo_q = "What was CVX's total net revenue for fiscal year 2023?"

result = baseline_rag(demo_q)

print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Retrieved {len(result['chunks'])} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  score={c['score']:.3f}")

chunks: [{'text': 'tivities and technology companies. \nRefer to “Cautionary Statements Relevant to Forward-Looking Information” on page 2 and to “Risk Factors” on pages 20 through 26 of the company’s 2022 Annual Report on Form 10-K for a discussion of some of the inherent risks that could materially impact the company’s results of operations or financial condition.\nNoteworthy Developments\nCertain noteworthy developments in recent months included the following:\n•\nUnited States – Announced an agreement to install new technologies on the company’s LNG vessels that are intended to reduce the carbon intensity of its LNG fleet operations.\n•\nUnited States – Announced an expansion of the Bayou Bend carbon capture and sequestration project in the U.S. Gulf Coast through an acquisition of nearly 100,000 acres of pore space, positioning Bayou Bend to become one of the largest carbon storage projects in the U.S.\n•\nUnited States – Announced commercial collaboration to purchase next generat

## 8. Evaluation (LLM-as-Judge)

Uses the same judge prompts as `langgraph_agentic_rag_sec_v2`.
- **Correctness** (0-1): does the answer match the reference on values, units, fiscal period?
- **Faithfulness** (0-1): is every claim grounded in the retrieved context?
- **Claims covered** (0-1): fraction of reference facts present in the candidate.
- **Refusal accuracy**: for adversarial questions, did the model correctly say it cannot answer?

In [19]:
eval_df = pd.read_csv(CONFIG["eval_path"])
eval_df = eval_df[eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

print(f"Evaluation split : '{CONFIG['eval_split']}'  ({len(eval_df)} questions)")
print(eval_df["question_type"].value_counts().to_string())

Evaluation split : 'dev'  (20 questions)
question_type
extractive     5
paraphrased    5
reasoning      5
adversarial    5


In [20]:
JUDGE_CORRECTNESS_SYSTEM = (
    "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
    "1 = correct value, correct units, correct period. "
    "0 = wrong number, wrong company, wrong period, or completely missing. "
    "Give partial credit for answers close but with unit errors. "
    "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
    "Return a valid JSON object only with keys: "
    "correctness (float 0-1), claims_covered (float 0-1), explanation (string)."
)

JUDGE_FAITHFULNESS_SYSTEM = (
    "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
    "1 = every number and claim is directly supported by the context. "
    "0 = answer contains numbers or claims not present in the context (hallucinated). "
    "Also set claims_covered: fraction of claims in the candidate supported by the context. "
    "Return a valid JSON object only with keys: "
    "faithfulness (float 0-1), claims_covered (float 0-1), explanation (string)."
)


def judge_correctness(question: str, reference: str, candidate: str) -> dict:
    user_msg = (
        f"Question:\n{question}\n\n"
        f"Reference answer:\n{reference}\n\n"
        f"Candidate answer:\n{candidate}"
    )
    raw = call_llm(
        messages=[
            {"role": "system", "content": JUDGE_CORRECTNESS_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        model=CONFIG["judge_model"],
        temperature=CONFIG["temperature_judge"],
        max_tokens=CONFIG["max_tokens_judge"],
        json_mode=True,
    )
    try:
        return json.loads(raw)
    except Exception:
        return {"correctness": 0.0, "claims_covered": 0.0, "explanation": "parse error"}


def judge_faithfulness(question: str, context: str, candidate: str) -> dict:
    user_msg = (
        f"Question:\n{question}\n\n"
        f"Retrieved context:\n{context[:3500]}\n\n"
        f"Candidate answer:\n{candidate}"
    )
    raw = call_llm(
        messages=[
            {"role": "system", "content": JUDGE_FAITHFULNESS_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        model=CONFIG["judge_model"],
        temperature=CONFIG["temperature_judge"],
        max_tokens=CONFIG["max_tokens_judge"],
        json_mode=True,
    )
    try:
        return json.loads(raw)
    except Exception:
        return {"faithfulness": 0.0, "claims_covered": 0.0, "explanation": "parse error"}

In [23]:
REFUSAL_KEYWORDS = [
    "insufficient", "not contain", "not available", "cannot find",
    "don't have", "no information", "not provided", "unable to",
    "not enough", "not present", "not found",
]

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating baseline"):
    question  = row["question"]
    reference = str(row["expected_answer"])

    rag = baseline_rag(question)
    answer  = rag["answer"]
    context = rag["context"]

    corr  = judge_correctness(question, reference, answer)
    faith = judge_faithfulness(question, context, answer)
    refused = any(kw in answer.lower() for kw in REFUSAL_KEYWORDS)

    results.append({
        "question_id":       row["question_id"],
        "question":          question,
        "question_type":     row["question_type"],
        "expected_decision": row.get("expected_decision", "ANSWER"),
        "expected_answer":   reference,
        "predicted_answer":  answer,
        "refused":           refused,
        "correctness":       float(corr.get("correctness", 0.0)),
        "faithfulness":      float(faith.get("faithfulness", 0.0)),
        "claims_covered":    float(corr.get("claims_covered", 0.0)),
        "n_chunks":          len(rag["chunks"]),
        "corr_explanation":  corr.get("explanation", ""),
    })
    time.sleep(CONFIG["inter_question_sleep_sec"])

results_df = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "baseline_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

Evaluating baseline: 100%|██████████| 20/20 [09:08<00:00, 27.44s/it]

Saved 20 rows -> results\baseline_results.csv


## 9. Results

In [24]:
print("=" * 55)
print(" BASELINE RAG -- EVALUATION RESULTS")
print("=" * 55)

print(f"\nOverall (n={len(results_df)}):")
for col in ["correctness", "faithfulness", "claims_covered"]:
    print(f"  {col:20s}: {results_df[col].mean():.3f}")

print("\nBy question type:")
by_type = (
    results_df
    .groupby("question_type")[["correctness", "faithfulness", "claims_covered"]]
    .mean()
    .round(3)
)
print(by_type.to_string())

adv = results_df[
    (results_df["question_type"] == "adversarial") &
    (results_df["expected_decision"] == "REFUSE")
]
if len(adv):
    print(f"\nAdversarial refusal accuracy: {adv['refused'].mean():.2%}  ({adv['refused'].sum()}/{len(adv)})")

print("\nBottom-5 by correctness:")
worst = results_df.nsmallest(5, "correctness")[
    ["question_id", "question_type", "correctness", "corr_explanation"]
]
print(worst.to_string(index=False))

 BASELINE RAG -- EVALUATION RESULTS

Overall (n=20):
  correctness         : 0.225
  faithfulness        : 0.980
  claims_covered      : 0.000

By question type:
               correctness  faithfulness  claims_covered
question_type                                           
adversarial            0.9          1.00             0.0
extractive             0.0          1.00             0.0
paraphrased            0.0          1.00             0.0
reasoning              0.0          0.92             0.0

Adversarial refusal accuracy: 20.00%  (1/5)

Bottom-5 by correctness:
 question_id question_type  correctness                                                                                                                                                                                                                                                                                                                                                                                                  

In [25]:
results_df

,question_id,question,question_type,expected_decision,expected_answer,predicted_answer,refused,correctness,faithfulness,claims_covered,n_chunks,corr_explanation
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,ANSWER,Apple's total net sales in fiscal year 2023 we...,"Unfortunately, the provided context does not c...",True,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",I cannot provide information on countries subj...,False,0.0,1.0,0.0,5,Candidate answer does not provide any informat...
2,3.0,What are the contingencies declared by Microso...,extractive,ANSWER,Microsoft declared a contingency related to IR...,I couldn't find any information about Microsof...,False,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
3,4.0,Which Microsoft business segment generated the...,extractive,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, I am unable to ...",True,0.0,1.0,0.0,5,The candidate answer is incorrect as it does n...
4,5.0,What risk does Nvidia mention that could affec...,extractive,ANSWER,Customers may delay adopting new architectures...,I couldn't find any information about Nvidia i...,False,0.0,1.0,0.0,5,The candidate answer is incorrect because it m...
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,ANSWER,Apple's total net sales in fiscal year 2023 we...,I cannot provide information on Apple's revenu...,False,0.0,1.0,0.0,5,Candidate answer is completely missing the req...
6,27.0,Which regions were impacted by the newly annou...,paraphrased,ANSWER,"According to Apple's Q3 2025 filing, the U.S. ...",I cannot provide information on the newly anno...,False,0.0,1.0,0.0,5,The candidate answer does not provide any info...
7,28.0,What potential financial obligations did Micro...,paraphrased,ANSWER,Microsoft declared a contingency related to IR...,"Based on the provided context, I couldn't find...",False,0.0,1.0,0.0,5,The candidate answer is incorrect because it d...
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,ANSWER,Intelligent Cloud generated the highest revenu...,"Based on the provided context, I am unable to ...",True,0.0,1.0,0.0,5,The candidate answer is completely incorrect a...
9,30.0,What factor does Nvidia identify that could de...,paraphrased,ANSWER,Customers may delay adopting new architectures...,I couldn't find any information in the provide...,False,0.0,1.0,0.0,5,The candidate answer is completely off-topic a...
